# MT07 — Cross-Domain RL with QAvatar

### Lab Description

Training a control policy from scratch is expensive. **Cross-Domain Reinforcement Learning (CDRL)** asks whether a policy already trained in a *source* domain (say, a 4-leg ant, or a Panda arm) can accelerate learning in a *different* but related *target* domain (a 5-leg ant, or a UR5e arm) — even when the state and action spaces do not match.

This lab uses **QAvatar**, which learns a small *decoder* from the target domain into the source domain and reuses the frozen source critic through a pair of normalizing-flow (RealNVP) maps. We compare QAvatar against a plain **SAC** baseline trained only on the target. To keep the notebook runnable end-to-end, the in-notebook training is a **short demo** (a few hundred–thousand steps) that only shows the mechanics and partial learning curves; the **side-by-side rollouts load fully-trained (baked) checkpoints** shipped with the course image.

#### Recommended Hardware
AMD Ryzen™ AI Halo Processors (e.g., AI Max+ 395, AI Max 390)
#### Software Environment
OS: Ubuntu 24.04.4 LTS \
Install [AUP learning cloud](https://amdresearch.github.io/aup-learning-cloud/installation/quick-start.html?family=ryzen-ai&gpu=max-pro-395). After installing AUP Learning Cloud you will have the ROCm + PyTorch environment (the `auplc-mujoco-torch` course image: torch + mujoco + robosuite + gymnasium) that this notebook is built for. The CDRL implementation (`cdrl_code/`) and pretrained weights (`cdrl_assets/`) ship next to this notebook.

## Goals
- Understand the CDRL setup: source vs. target domains and what QAvatar transfers
- Build source/target environments and initialize QAvatar and an SAC baseline
- Run a short in-notebook demo train and read the learning curves
- Load fully-trained checkpoints and compare QAvatar vs. SAC rollouts side by side
  - **Part A** — locomotion: 4-leg → 5-leg Ant (MuJoCo)
  - **Part B** — manipulation: Panda → UR5e on the Door task (robosuite)

### Imports and paths

`cdrl_code/` holds the CDRL implementation (`utils.py`, `SAC.py`, `QAvatar.py`, and the `core/flow` RealNVP modules); we put it on `sys.path`. `cdrl_assets/` holds the pretrained **source** models + flows and the fully-trained **target** checkpoints used for the rollouts. We force the EGL GL backend, which is the reliable offscreen path on this ROCm/gfx1151 stack.

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

import sys
import numpy as np
import torch
import matplotlib.pyplot as plt
import imageio
import mujoco
from argparse import Namespace
from tqdm import tqdm
from PIL import Image, ImageDraw
from IPython.display import Video

import logging
logging.getLogger("robosuite_logs").setLevel(logging.ERROR)

# CDRL implementation + baked weights ship next to this notebook
CDRL_CODE = os.path.abspath("cdrl_code")
CDRL_ASSETS = os.path.abspath("cdrl_assets")
sys.path.insert(0, CDRL_CODE)

from utils import seed_everything, EvalManager, construct_env  # noqa: E402
from SAC import SAC  # noqa: E402
from QAvatar import QAvatar  # noqa: E402

os.makedirs("output/videos", exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| torch", torch.__version__, "| mujoco", mujoco.__version__)

### Shared helpers

Both parts reuse three helpers, defined one per cell below.

**Helper 1 — short demo training.** `run_demo_training` is a trimmed off-policy loop (collect → store → update → periodic eval) meant only to expose the mechanics and produce partial curves. Evaluation runs quietly (`verbose=False`) and the progress bar is a single fixed-width line.

In [ ]:
def run_demo_training(agent, args, envs, demo_timesteps):
    """Short in-notebook training — mechanics + partial curves only.
    Full target policies are loaded from baked checkpoints for the rollouts."""
    seed_everything(args.seed)
    envs.action_space.seed(args.seed)
    eval_env = EvalManager(construct_env(args.tar_env, args.seed)[0], device=device,
                           reset_seed=args.seed, eval_episodes=2)
    obs, _ = envs.reset(seed=args.seed)
    obs = torch.as_tensor(obs, dtype=torch.float32, device=device)
    for step in tqdm(range(1, demo_timesteps + 1), desc="demo", ncols=100):
        if step <= args.learning_starts:
            a = envs.action_space.sample()
            actions = torch.as_tensor(a, dtype=torch.float32, device=device).unsqueeze(0)
        else:
            with torch.no_grad():
                actions, _ = agent.get_action(obs.unsqueeze(0))
        next_obs, r, term, trunc, _ = envs.step(actions[0].cpu().numpy())
        next_obs = torch.as_tensor(next_obs, dtype=torch.float32, device=device)
        agent.store_data(obs, next_obs, actions, r, term, trunc)
        if term or trunc:
            next_obs, _ = envs.reset()
            next_obs = torch.as_tensor(next_obs, dtype=torch.float32, device=device)
        obs = next_obs
        agent.train(step, demo_timesteps + 1)
        if step % args.evaluate_freq == 0:
            eval_env.evaluate(agent, step, verbose=False)
    return eval_env.results

**Helper 2 — training & evaluation curves.** `plot_curves` draws the six standard SAC/QAvatar diagnostics (Q loss, actor loss, alpha, alpha loss, and the periodic evaluation reward/length) for the two agents on shared axes.

In [ ]:
def plot_curves(agent_q, agent_s, q_res, s_res, suffix=""):
    plt.figure(figsize=(16, 8))
    panels = [("Q Loss", agent_q.Q_losses, agent_s.Q_losses),
              ("Actor Loss", agent_q.actor_losses, agent_s.actor_losses),
              ("Alpha", agent_q.alphas, agent_s.alphas),
              ("Alpha Loss", agent_q.alpha_losses, agent_s.alpha_losses)]
    for i, (name, q, s) in enumerate(panels, 1):
        plt.subplot(2, 3, i)
        plt.plot(q, label="QAvatar"); plt.plot(s, label="SAC")
        plt.title(name); plt.xlabel("update step"); plt.legend()
    qk, sk = sorted(q_res), sorted(s_res)
    plt.subplot(2, 3, 5)
    plt.plot(qk, [np.mean(q_res[k]["rewards"]) for k in qk], label="QAvatar")
    plt.plot(sk, [np.mean(s_res[k]["rewards"]) for k in sk], label="SAC")
    plt.title("Eval Reward"); plt.xlabel("env step"); plt.legend()
    plt.subplot(2, 3, 6)
    plt.plot(qk, [np.mean(q_res[k]["lengths"]) for k in qk], label="QAvatar")
    plt.plot(sk, [np.mean(s_res[k]["lengths"]) for k in sk], label="SAC")
    plt.title("Eval Length"); plt.xlabel("env step"); plt.legend()
    plt.suptitle("Short in-notebook demo curves" + suffix)
    plt.tight_layout(); plt.show(); plt.close()

**Helper 3 — side-by-side rollout video.** `rollout_compare` rolls both agents in the target env and writes a side-by-side `.mp4`. MuJoCo uses the gym renderer; robosuite is rendered through a direct `mujoco.Renderer` on geom group 1 (`_make_rs_renderer` / `_rs_frame`) to avoid the corrupted frames the built-in camera path produces on this ROCm/gfx1151 stack. `_label` overlays the running return on each panel.

In [ ]:
def _label(frame, text):
    img = Image.fromarray(frame)
    ImageDraw.Draw(img).text((12, 10), text, fill=(255, 255, 255))
    return np.array(img)


def _make_rs_renderer(env, h=384, w=384):
    sim = getattr(env, "sim", None) or env.unwrapped.sim
    R = mujoco.Renderer(sim.model._model, height=h, width=w)
    opt = mujoco.MjvOption()
    opt.geomgroup[:] = 0
    opt.geomgroup[1] = 1
    return R, opt, sim


def _rs_frame(R, opt, sim, camera="frontview"):
    mujoco.mj_forward(sim.model._model, sim.data._data)
    R.update_scene(sim.data._data, camera=camera, scene_option=opt)
    return R.render()


def rollout_compare(agent_q, agent_s, args, out_path, max_steps=300, fps=30, camera="frontview"):
    q_env, env_type = construct_env(args.tar_env, args.seed, render_mode="rgb_array")
    s_env, _ = construct_env(args.tar_env, args.seed, render_mode="rgb_array")
    rs = env_type == "robosuite"
    if rs:
        qR, qopt, qsim = _make_rs_renderer(q_env)
        sR, sopt, ssim = _make_rs_renderer(s_env)
    q_obs, _ = q_env.reset(seed=args.seed)
    s_obs, _ = s_env.reset(seed=args.seed)
    q_obs = torch.as_tensor(q_obs, dtype=torch.float32, device=device)
    s_obs = torch.as_tensor(s_obs, dtype=torch.float32, device=device)
    q_ret = s_ret = 0.0
    q_done = s_done = False
    qf = sf = None
    frames = []
    for _ in tqdm(range(max_steps), desc="rollout", ncols=100):
        if not q_done:
            qf = q_env.render() if not rs else _rs_frame(qR, qopt, qsim, camera)
            with torch.no_grad():
                qa = agent_q.get_action(q_obs.unsqueeze(0), test=True)
            no, r, te, tr, _ = q_env.step(qa[0].cpu().numpy())
            q_obs = torch.as_tensor(no, dtype=torch.float32, device=device)
            q_ret += float(r); q_done = te or tr
        if not s_done:
            sf = s_env.render() if not rs else _rs_frame(sR, sopt, ssim, camera)
            with torch.no_grad():
                sa = agent_s.get_action(s_obs.unsqueeze(0), test=True)
            no, r, te, tr, _ = s_env.step(sa[0].cpu().numpy())
            s_obs = torch.as_tensor(no, dtype=torch.float32, device=device)
            s_ret += float(r); s_done = te or tr
        h = min(qf.shape[0], sf.shape[0]); w = min(qf.shape[1], sf.shape[1])
        left = _label(np.ascontiguousarray(qf[:h, :w]), f"QAvatar  return={q_ret:.1f}")
        right = _label(np.ascontiguousarray(sf[:h, :w]), f"SAC  return={s_ret:.1f}")
        frames.append(np.concatenate([left, right], axis=1))
    q_env.close(); s_env.close()
    if rs:
        qR.close(); sR.close()
    imageio.mimsave(out_path, frames, fps=fps)
    print(f"saved {out_path} | QAvatar return={q_ret:.1f} | SAC return={s_ret:.1f}")
    return out_path

## Part A — Locomotion transfer: 4-leg → 5-leg Ant

The source domain is Gymnasium's standard `Ant-v5` (4 legs); the target is a modified 5-leg ant (extra limb → larger state/action space). QAvatar reuses the frozen 4-leg critic via its RealNVP flows and a learned decoder, while SAC starts from scratch on the 5-leg ant.

### Configuration

`total_timesteps` is the *full* training budget (kept for reference); the notebook actually runs only `DEMO_STEPS_ANT` steps in-place. `src_models_path` points at the baked 4-leg source model + flows. `nn_hidden_size=256` matches the size the source/target checkpoints were trained with.

In [ ]:
ant_args = Namespace(
    seed=0, total_timesteps=300000, buffer_size=20000, learning_starts=100,
    q_lr=3e-4, policy_lr=3e-4, decoder_lr=1e-4, nn_hidden_size=256, batch_size=256,
    gamma=0.99, temperature_opt=True, evaluate_freq=500, tau=0.005,
    save_model_interval=[], cuda=True,
    tar_env="5leg_ant", src_envs=["4leg_ant"],
    src_models_path=[os.path.join(CDRL_ASSETS, "source_model", "4leg_ant") + os.sep],
)
DEMO_STEPS_ANT = 1500  # short demo; the real policies come from baked checkpoints

ant_env, _ = construct_env(ant_args.tar_env, ant_args.seed)
ant_src = [construct_env(e, ant_args.seed)[0] for e in ant_args.src_envs]
ant_q = QAvatar(ant_args, ant_src, ant_env, device)
ant_s = SAC(ant_args, ant_env.observation_space.shape[0], ant_env.action_space.shape[0],
            ant_env.action_space.low, ant_env.action_space.high, device)
print("target obs/act:", ant_env.observation_space.shape, ant_env.action_space.shape)

### Short demo training + learning curves

We run both agents for `DEMO_STEPS_ANT` steps and plot the six diagnostics. With such a short budget the curves are noisy and neither policy is finished — this cell is about *seeing the loop run*, not final performance.

In [ ]:
ant_q_res = run_demo_training(ant_q, ant_args, ant_env, DEMO_STEPS_ANT)
ant_s_res = run_demo_training(ant_s, ant_args, ant_env, DEMO_STEPS_ANT)
plot_curves(ant_q, ant_s, ant_q_res, ant_s_res, " — Ant 4→5 leg")

### Rollout: fully-trained QAvatar vs. SAC (300k steps)

We overwrite the demo weights with the **baked 300k-step checkpoints** for both agents and render a side-by-side rollout in the 5-leg ant. This is the fair, fully-trained comparison.

In [ ]:
ant_q.load(os.path.join(CDRL_ASSETS, "checkpoints", "ant_4leg_to_5leg", "QAvatar", "steps_300000.pt"))
ant_s.load(os.path.join(CDRL_ASSETS, "checkpoints", "ant_4leg_to_5leg", "SAC", "steps_300000.pt"))
rollout_compare(ant_q, ant_s, ant_args, "output/videos/mt07_ant_4to5.mp4", max_steps=300)

In [ ]:
Video(url="output/videos/mt07_ant_4to5.mp4")

## Part B — Manipulation transfer: Panda → UR5e (Door)

Same idea, harder domain: transfer from a Panda arm to a UR5e arm on robosuite's **Door** task (turn the handle and open the door — same task, different robot morphology, the natural continuation of MT01–MT05). `nn_hidden_size=128` matches the robosuite source/target checkpoints. Unlike Part A's 300k budget, this pair was trained to **500k steps**, so the rollout below is a fully-trained, fair QAvatar vs. SAC comparison.

In [ ]:
door_args = Namespace(
    seed=4, total_timesteps=500000, buffer_size=20000, learning_starts=200,
    q_lr=1e-4, policy_lr=1e-4, decoder_lr=1e-4, nn_hidden_size=128, batch_size=256,
    gamma=0.99, temperature_opt=True, evaluate_freq=400, tau=0.005,
    save_model_interval=[], cuda=True,
    tar_env="Door_UR5e", src_envs=["Door_Panda"],
    src_models_path=[os.path.join(CDRL_ASSETS, "source_model", "Door_Panda") + os.sep],
)
DEMO_STEPS_DOOR = 800

door_env, _ = construct_env(door_args.tar_env, door_args.seed)
door_src = [construct_env(e, door_args.seed)[0] for e in door_args.src_envs]
door_q = QAvatar(door_args, door_src, door_env, device)
door_s = SAC(door_args, door_env.observation_space.shape[0], door_env.action_space.shape[0],
             door_env.action_space.low, door_env.action_space.high, device)
print("target obs/act:", door_env.observation_space.shape, door_env.action_space.shape)

In [ ]:
door_q_res = run_demo_training(door_q, door_args, door_env, DEMO_STEPS_DOOR)
door_s_res = run_demo_training(door_s, door_args, door_env, DEMO_STEPS_DOOR)
plot_curves(door_q, door_s, door_q_res, door_s_res, " — Door Panda→UR5e")

### Rollout: fully-trained QAvatar vs. SAC (500k steps)

We overwrite the demo weights with the baked **500k-step** checkpoints for both agents and render the Door rollout via the direct `mujoco.Renderer` path (geom group 1). This is the fair, fully-trained comparison.

In [ ]:
door_q.load(os.path.join(CDRL_ASSETS, "checkpoints", "door_panda_to_ur5e", "QAvatar", "steps_500000.pt"))
door_s.load(os.path.join(CDRL_ASSETS, "checkpoints", "door_panda_to_ur5e", "SAC", "steps_500000.pt"))
rollout_compare(door_q, door_s, door_args, "output/videos/mt07_door_panda_to_ur5e.mp4",
                max_steps=500, camera="frontview")

In [ ]:
Video(url="output/videos/mt07_door_panda_to_ur5e.mp4")

## Conclusions

You set up a cross-domain RL experiment and compared **QAvatar** — which reuses a frozen source-domain critic through learned RealNVP flows and a decoder — against a from-scratch **SAC** baseline, on both a locomotion morphology change (4→5 leg ant) and a manipulation robot swap (Panda→UR5e on the Door task). The short in-notebook training shows the off-policy loop and its diagnostics, while the baked fully-trained checkpoints (Part A at 300k, Part B at 500k) show the finished behavior: transferring source knowledge lets QAvatar reach competent target behavior faster than the from-scratch SAC baseline, in both the locomotion and manipulation settings.

## Acknowledgements

This notebook was developed in collaboration with the lab of Professor Ping-Chun Hsieh (謝秉均), Department of Computer Science, National Yang Ming Chiao Tung University (NYCU) — [pinghsieh.github.io](https://pinghsieh.github.io/).

---

Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved. Portions of this file consist of AI-generated content.
SPDX-License-Identifier: MIT